# Notebook for configuring jobs and processing
#### Fist we export the needed

In [1]:
import numpy as np
import pickle, sys, os, json, glob, tables, subprocess
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import pandas as pd

sys.path.insert(0, os.path.join(os.getcwd(), "../scripts/"))
import utils

# --- Version --- #
import lstchain
root_lstchain = lstchain.__path__[0]
version_lstchain = f"v{(lstchain.__version__).split('.dev')[0]}"
print(f"Using lstchain version {version_lstchain} from:\n{root_lstchain}")

Using lstchain version v0.10.18 from:
/fefs/aswg/workspace/juan.jimenez/softs/analysis/cta-lstchain/lstchain


# <span style="color:blue">1. Configuring file for each job to be sent</span>

In [2]:
"""Number of subruns in one job"""
n_subruns_job = 5

### Paths and filenames

In [3]:
# Root path of this script
root = os.getcwd() + "/"
# Path to store the configuration file we are going to use
root_config = root + "config/"

#####################################################
file_config_job = os.path.join(root_config, "config_jobs_runs.txt")
file_config_scaling = os.path.join(root_config, "config_scaling_parameters.json")
file_config_lstchain_dl2 = os.path.join(root_config, "config_lstchain_dl2.json")
file_config_lstchain_dl3 = os.path.join(root_config, "config_lstchain_dl3.json")

# STANDARD paths ---------
root_dl1 = "/fefs/aswg/data/real/DL1/????????/v*/tailcut*/"
root_dl2 = "/fefs/aswg/data/real/DL2/????????/v*/tailcut*/nsb_tuning_*/"
default_config_lstchain_dl3 = os.path.join(root_lstchain, "../docs/examples/irf_dl3_tool_config.json")

# Create the path for the config files
os.makedirs(root_config, exist_ok=True)

# Some options
overwrite = True
simulate_data = False
process_inline = False
process_all = True

### Run numbers we have

In [4]:
obs_ids = [19799, 19800, 19801, 19802, 19803, 19804, 19805, 19806, 19807, 19808, 19809, 19810, 19811]

print(f"Computing for {len(obs_ids)} runs")

Computing for 13 runs


### Reading the number of subruns from the datachecks

In [5]:
%%time
dict_run_sruns, dict_telapsed, dict_tstamp, dict_zd = {}, {}, {}, {}
for obs_id in obs_ids:
    query_dcheck = glob.glob(root_dl1 + f"datacheck/*Run{obs_id}.h5")
    
    if len(query_dcheck) == 0:
        print(f"ERROR: No datacheck found for Run {obs_id}"); sys.exit()
    
    tab = tables.open_file(query_dcheck[0]).root.dl1datacheck.cosmics
    dict_run_sruns[obs_id] = [int(i) for i in tab.col("subrun_index")]
    elapsed_times = tab.col("elapsed_time")
    timestamps = tab.col("dragon_time")
    mean_zd_tel = np.pi / 2 - np.array(tab.col("mean_alt_tel"))
    
    dict_telapsed[obs_id], dict_zd[obs_id], dict_tstamp[obs_id] = {}, {}, {}
    for s, srun in enumerate(dict_run_sruns[obs_id]):
        dict_tstamp[obs_id][srun] = timestamps[s][0]
        dict_telapsed[obs_id][srun] = elapsed_times[s]
        dict_zd[obs_id][srun] = mean_zd_tel[s]
        
    tab.close()

CPU times: user 25.5 s, sys: 11 s, total: 36.5 s
Wall time: 49.9 s


### Storing the subrun numbers in sets of certain amount of subruns inside the same job

In [6]:
n_jobs = 0
with open(file_config_job, "w") as file:
    for obs_id in obs_ids:
        count_sruns = 0
        sruns = np.sort(dict_run_sruns[obs_id])
        tmp_str = ""
        for srun in sruns:
            tmp_str = tmp_str + f"_{srun}"
            # Launching a certain amount of subruns together
            if (count_sruns % n_subruns_job == 0 and srun != 0) or (srun == max(sruns)):
                tmp_str_splitted = tmp_str.split("_")
                if len(tmp_str_splitted) != 2:
                    tmp_str = "_" + tmp_str_splitted[1] + "_" + tmp_str_splitted[-1]
                file.write(f"{obs_id}{tmp_str}\n")
                tmp_str = ""
                n_jobs += 1
            count_sruns += 1
print(f"The final amount of jobs is {n_jobs * 2 + 1}")

# Reading the config file
jobs_list = np.atleast_1d(np.loadtxt(file_config_job, dtype="str"))

The final amount of jobs is 329


# <span style="color:blue">2. Configuring file for the fit parameters and etc</span>
#### Is stored permanently inside `config/config_scaling_parameters.json`

`dec_2276`,  `dec_3476`, `dec_4822`, `dec_6166`, `dec_6676`, `dec_931`, `dec_min_1802`, `dec_min_2924`, `dec_min_413`

In [7]:
# Permanent loczation of the configuration file
fname_config_scaling = os.path.join(root_config, "config_scaling_parameters.json")

""" Source name in order to just complete the results file, and in order to improve run organization."""
source_name = "S241125n"
str_dec = "dec_6676"

"""Reference intensity, center of the fit"""
ref_intensity = 422 # p.e.

""" Fit parameters
Chosen limits in intensity (p.e.) for applying the fit i.e. the power law will be fitted only with the 
points within this range."""
c = [316, 562]
""" For the positive scaling cases (most of them), we need to have a lower  limit in intensity. This 
limit is used for the subset of events that are scaled just to find which is the scaling value. We use a 
very low limit by default 60 p.e. compared to the lower limit of the fit 316 p.e. because in the worst 
cases we will have a very non-linear scaling that will displace significantly the events intensities."""
lims_intensity = [316, 562]
lims_intensity_extended = 50

"""Binning for intensity histograms and fits. We can use the same as is used in the datachecks by default."""
binning_intensity = list(np.logspace(1, 6, 101))

""" Power law parameters for the reference
All these parameters are taken from a common analysis of the full dataset Where the period of end 
of 2022 and start 2023 is taken as reference for good runs. Then we take as reference the mean 
power law parameters in that period. p0 is the normalization factor and p1 is the slope."""
ref_p0 =  1.74 
ref_p1 = -2.23

""" Threshold in statistics for the last subrun
The limit in number of events after cleaning that we need to consider the last subrun has enough 
statistics to perform the analysis over it. Otherwise the values of the scaling that will be applied 
to this last rubrun are the same that are applied to the last last subrun."""
statistics_threshold = 4000

""" The number of tries in the dl1 files creation. Due that dl1 file creation have some bug that causes the 
file to not be present but not notify with any error. We implemented a loop and a check to see if 
the file exists. """
number_tries_dl1 = 3

""" Parameters for the empyrical fits for Zenith Distance corrections Are simply two 2 degree polynomials 
for each variable 
of the power law."""
p0a, p0b, p0c = -0.44751321, 3.62502037, -1.43611437
p1a, p1b, p1c = -2.89253919, 0.99443581, -0.34013068

# Standard wildcards for data in the IT cluster ---------
root_dl1 = "/fefs/aswg/data/real/DL1/????????/v*/tailcut*/"
root_dl2 = "/fefs/aswg/data/real/DL2/????????/v*/tailcut*/nsb_tuning_*/"
root_rfs = "/fefs/aswg/data/models/AllSky/20240918_v0.10.12_allsky_nsb_tuning_*/"
root_mcs = "/fefs/aswg/data/mc/DL2/AllSky/20240918_v0.10.12_allsky_nsb_tuning_*/"

root            = os.getcwd() # Root path of this script
root_slurm      = os.path.join(root, "objects","output_slurm") # To store the slurm outputs
root_tmp_plots  = os.path.join(root, "objects","tmp_plots")
root_scripts    = os.path.join(root, "..", "scripts")
file_config_job = file_config_job # Config file for jobs
file_config_lstchain_dl2 = file_config_lstchain_dl2 # Path to store the config
file_config_lstchain_dl3 = file_config_lstchain_dl3

root_objects       = os.path.join(root, "objects") # Path to store objects
root_sub_dl1       = os.path.join(root_objects, "sub_dl1") # Sub-dl1 objects directory
root_results       = os.path.join(root_objects, "results_fits") # Results
root_results_final = os.path.join(root_objects, "results_fits_final") # Merged results

# Directories for the data
# Main directory
root_data = os.path.join(
    "/fefs/aswg/workspace/juan.jimenez/data/", "real", "mono",
    f"{source_name}", version_lstchain, "GammaDiffuse", "prod_light_scaling"
)

dir_dl1 = os.path.join(root_data, "DL1")

### Insert in a dictionary

In [8]:
dict_config = {
    "source_name": source_name, "str_dec": str_dec, "obs_ids": obs_ids,
    
    "dicts": {
        "sruns": dict_run_sruns, "telapsed": dict_telapsed, "zd": dict_zd,
    },
    
    "ref_intensity": ref_intensity,

    "lims_intensity": lims_intensity, "lims_intensity_extended": lims_intensity_extended,

    "ref_p0": ref_p0, "ref_p1": ref_p1,
    "p0a": p0a, "p0b": p0b, "p0c": p0c,
    "p1a": p1a, "p1b": p1b, "p1c": p1c,

    "stats_th": statistics_threshold, "number_tries_dl1": number_tries_dl1,

    "binning_intensity": binning_intensity,

    "root_dl1": root_dl1, "root_dl2": root_dl2, "root_rfs": root_rfs, "root_mcs": root_mcs,

    "root": root, "root_slurm": root_slurm, "root_tmp_plots": root_tmp_plots,
    "root_scripts": root_scripts, "root_objects": root_objects,
    "root_results": root_results, "root_results_final": root_results_final,
    "root_data": root_data, "root_sub_dl1": root_sub_dl1,

    "file_config_lstchain_dl2": file_config_lstchain_dl2,
    "file_config_lstchain_dl3": file_config_lstchain_dl3,
    "file_config_job": file_config_job,
    
    "dir_dl1": dir_dl1,
}

# Store in a file
# Open the file in write mode
with open(file_config_scaling, "w") as json_file:
    json.dump(dict_config, json_file)

##### Create the folders

In [9]:
# Create the paths that do not exist
for path in [root_objects, root_slurm, root_objects, root_sub_dl1, root_results, 
             root_results_final, dir_dl1, root_tmp_plots]:
    os.makedirs(path, exist_ok=True)

# <span style="color:blue">3. Sending the jobs (or running)</span>
##### <span style="color:blue">3.1 Init: Subrun iterative process to get srun-scaling factors</span>

In [ ]:
%%time
if process_all: 

    for i, job in enumerate(jobs_list[:]):
        i = list(jobs_list).index(job)
        print(f"\nGetting scaling factor for job \"{job}\" {i}/{len(jobs_list)}...\n")      

        # Then we call the init script
        str_args = f"{job} {file_config_scaling} {simulate_data}"
        python_command = f"python script_1_scaling.py main_init {str_args}"

        str_output = f"-o ./objects/output_slurm/scaling_init_{job}.out"
        slurm_command = f"sbatch -p short -J scaling_init_{job} {str_output} --wrap='{python_command}'"

        command = python_command if process_inline else slurm_command
        subprocess.run(command, shell=True, text=True)

In [ ]:
!squeue -u juan.jimenez
# !scancel -u juan.jimenez 

In [ ]:
if process_all:
    num_errors = 0
    for i, job in enumerate(jobs_list[:]):
        file_out = f"./objects/output_slurm/scaling_init_{job}.out"
        with open(file_out, 'r') as f:
            if not "SUCCESFULLY FINISHED" in f.read():
                print(f"ERROR/RUNNING found in job: {job}")
                num_errors += 1
    print(f"\nTotal amount of failed/running jobs {num_errors} / {len(jobs_list)}")

##### <span style="color:blue">3.1 Merge: Merging results `.pkl` files.</span>

In [ ]:
if process_all:
    python_command = f"python script_1_scaling.py main_merge {file_config_scaling}"
    subprocess.run(python_command, shell=True, text=True)

##### <span style="color:blue">3.3 Final: Scaling with final values</span>

In [10]:
%%time
if process_all:

    for i, job in enumerate(jobs_list[:]):
        i = list(jobs_list).index(job)
        print(f"\nApplying definitive scaling factor for \"{job}\" {i}/{len(jobs_list)}...\n")      

        # Then we call the final script
        str_args = f"{job} {file_config_scaling} {simulate_data}"
        python_command = f"python script_1_scaling.py main_final {str_args}"

        str_output = f"-o ./objects/output_slurm/scaling_final_{job}.out"
        slurm_command = f"sbatch -p short --mem=10000 -J scaling_final_{job} {str_output} --wrap='{python_command}'"

        command = python_command if process_inline else slurm_command
        subprocess.run(command, shell=True, text=True)



Applying definitive scaling factor for "19799_0_5" 0/164...

Submitted batch job 44271015

Applying definitive scaling factor for "19799_6_10" 1/164...

Submitted batch job 44271016

Applying definitive scaling factor for "19799_11_15" 2/164...

Submitted batch job 44271017

Applying definitive scaling factor for "19799_16_20" 3/164...

Submitted batch job 44271018

Applying definitive scaling factor for "19799_21_25" 4/164...

Submitted batch job 44271019

Applying definitive scaling factor for "19799_26_30" 5/164...

Submitted batch job 44271020

Applying definitive scaling factor for "19799_31_35" 6/164...

Submitted batch job 44271021

Applying definitive scaling factor for "19799_36_40" 7/164...

Submitted batch job 44271022

Applying definitive scaling factor for "19799_41_45" 8/164...

Submitted batch job 44271023

Applying definitive scaling factor for "19799_46_50" 9/164...

Submitted batch job 44271024

Applying definitive scaling factor for "19799_51_55" 10/164...

Submitte

Submitted batch job 44271103

Applying definitive scaling factor for "19805_11_15" 89/164...

Submitted batch job 44271104

Applying definitive scaling factor for "19805_16_20" 90/164...

Submitted batch job 44271105

Applying definitive scaling factor for "19805_21_25" 91/164...

Submitted batch job 44271106

Applying definitive scaling factor for "19805_26_30" 92/164...

Submitted batch job 44271107

Applying definitive scaling factor for "19805_31_35" 93/164...

Submitted batch job 44271108

Applying definitive scaling factor for "19805_36_40" 94/164...

Submitted batch job 44271109

Applying definitive scaling factor for "19805_41_45" 95/164...

Submitted batch job 44271110

Applying definitive scaling factor for "19805_46_50" 96/164...

Submitted batch job 44271111

Applying definitive scaling factor for "19805_51_55" 97/164...

Submitted batch job 44271112

Applying definitive scaling factor for "19805_56_60" 98/164...

Submitted batch job 44271113

Applying definitive scaling fa

In [36]:
!squeue -u juan.jimenez
# !scancel -u juan.jimenez

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON) 
          44271846     short dl1_to_d juan.jim CG       0:00      1 cp44 
          44271500     short dl2_mc_t juan.jim CG       0:00      1 cp05 


In [37]:
if process_all:
    num_errors = 0
    for i, job in enumerate(jobs_list[:]):
        file_out = f"./objects/output_slurm/scaling_final_{job}.out"
        with open(file_out, 'r') as f:
            if not "SUCCESFULLY FINISHED" in f.read():
                print(f"ERROR/RUNNING found in job: {job}")
                num_errors += 1

    print(f"\nTotal amount of failed/running jobs {num_errors} / {len(jobs_list)}")


Total amount of failed/running jobs 0 / 164


##### <span style="color:blue">3.4 Merge: Merging the final results `.pkl`</span>

In [38]:
if process_all:
    python_command = f"python script_1_scaling.py main_merge {file_config_scaling}"
    subprocess.run(python_command, shell=True, text=True)


For run 19799 subrun 26:
N_events = 2042
Flag_error = False
So interpolating neighbors.

For run 19799 subrun 27:
N_events = 2317
Flag_error = True
So interpolating neighbors.

For run 19800 subrun 6:
N_events = 3573
Flag_error = False
So interpolating neighbors.

For run 19805 subrun 50:
N_events = 18947
Flag_error = True
So interpolating neighbors.

For run 19807 subrun 37:
N_events = 2362
Flag_error = False
So interpolating neighbors.
Interpolating... Run19799 - 0.0%
Interpolating... Run19800 - 7.7%
Interpolating... Run19801 - 15.4%
Interpolating... Run19802 - 23.1%
Interpolating... Run19803 - 30.8%
Interpolating... Run19804 - 38.5%
Interpolating... Run19805 - 46.2%
Interpolating... Run19806 - 53.8%
Interpolating... Run19807 - 61.5%
Interpolating... Run19808 - 69.2%
Interpolating... Run19809 - 76.9%
Interpolating... Run19810 - 84.6%
Interpolating... Run19811 - 92.3%
SUCCESFULLY FINISHED


# <span style="color:blue">4. Merging DL1</span>

In [39]:
if process_all:
    from lstchain.scripts import lstchain_merge_hdf5_files
    
    for obs_id in obs_ids:
        print(f"Merging run {obs_id} ...")
        command  = "lstchain_merge_hdf5_files "
        command += f"--input-dir {os.path.join(dir_dl1, f'Run{obs_id}')} "
        command += f"--output-file {os.path.join(dir_dl1, f'dl1_LST-1.Run{obs_id}.h5')}"

        subprocess.run(command, shell=True, text=True)

Merging run 19799 ...


100%|██████████| 85/85 [00:38<00:00,  2.23it/s]


Merging run 19800 ...


100%|██████████| 18/18 [00:09<00:00,  1.92it/s]


Merging run 19801 ...


...

100%|██████████| 60/60 [00:26<00:00,  2.24it/s]


Merging run 19810 ...


100%|██████████| 59/59 [00:25<00:00,  2.35it/s]


Merging run 19811 ...


100%|██████████| 20/20 [00:08<00:00,  2.42it/s]
